# 三味線譜変換ツール

五線譜（PDF）→ MusicXML（Audiveris）→ 三味線中間YAML の変換を行います。

**実行順序:** セル0 → セル0-b → セル1-a → セル1-b → セル1-c → セル2 → セル3-a → セル3-b → セル3-c → セル4

In [ ]:
# ============================================================
# セル0: セットアップ（最初に一度だけ実行 / 再実行で最新版に更新）
# ============================================================

import subprocess, sys, os
from google.colab import userdata

REPO_NAME = "kamex120/shamisen-converter"
REPO_DIR  = "shamisen-converter"

def run(cmd, **kwargs):
    r = subprocess.run(cmd, capture_output=True, text=True, **kwargs)
    if r.stdout: print(r.stdout.strip())
    if r.returncode != 0:
        print("❌ エラー:", r.stderr.strip())
        raise RuntimeError(f"コマンド失敗: {' '.join(cmd)}")
    return r

# リポジトリのクローン or 更新
token = userdata.get("GITHUB_TOKEN")
repo_url = f"https://{token}@github.com/{REPO_NAME}.git"
if not os.path.exists(REPO_DIR):
    print("📥 リポジトリをクローン中...")
    run(["git", "clone", repo_url, REPO_DIR])
else:
    print("🔄 最新版に更新中...")
    run(["git", "pull"], cwd=REPO_DIR)

os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print(f"📁 作業ディレクトリ: {os.getcwd()}")

# 依存ライブラリのインストール
print("\n📦 ライブラリをインストール中...")
run([sys.executable, "-m", "pip", "install", "music21", "pyyaml", "-q"])

print("\n✅ セットアップ完了")

In [ ]:
# ============================================================
# セル0-b: Audiveris セットアップ（最初に一度だけ実行）
# ============================================================

import subprocess, os

DEB_URL = "https://github.com/Audiveris/audiveris/releases/download/5.10.2/Audiveris-5.10.2-ubuntu22.04-x86_64.deb"
DEB_PATH = "/tmp/audiveris.deb"
KNOWN_BIN = "/opt/audiveris/bin/Audiveris"

def _find_audiveris():
    r = subprocess.run(["which", "Audiveris"], capture_output=True, text=True)
    if r.returncode == 0 and r.stdout.strip():
        return r.stdout.strip()
    if os.path.isfile(KNOWN_BIN):
        return KNOWN_BIN
    return None

AUDIVERIS_BIN = _find_audiveris()

if AUDIVERIS_BIN:
    print(f"✅ Audiveris インストール済み: {AUDIVERIS_BIN}")
else:
    print("🔧 xvfb をインストール中...")
    subprocess.run(["apt-get", "install", "-y", "xvfb", "-q"], capture_output=True)

    print("📥 Audiveris をダウンロード中...")
    subprocess.run(["wget", "-q", "--show-progress", DEB_URL, "-O", DEB_PATH], check=True)

    print("📦 インストール中...")
    subprocess.run(["dpkg", "-i", DEB_PATH])
    subprocess.run(["apt-get", "install", "-f", "-y", "-q"], capture_output=True)
    os.remove(DEB_PATH)

    AUDIVERIS_BIN = _find_audiveris()
    if AUDIVERIS_BIN:
        print(f"✅ Audiveris セットアップ完了: {AUDIVERIS_BIN}")
    else:
        print("❌ Audiveris バイナリが見つかりません。インストールログを確認してください。")

---
## Step 1: PDF → MusicXML

In [ ]:
# ============================================================
# セル1-a: PDFをアップロード
# ============================================================

from google.colab import files

print("PDFファイルを選択してください")
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print(f"\n📄 アップロード: {pdf_path}")

In [ ]:
# ============================================================
# セル1-a2: インタラクティブ白塗りツール
# マウスでドラッグして消したい範囲を選択、複数選択可
# ============================================================

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "pymupdf", "ipympl", "-q"], check=True)

# ipympl を有効化（%matplotlib widget 相当）
get_ipython().run_line_magic("matplotlib", "widget")

import fitz
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.widgets import RectangleSelector

SCALE = 1.5
selected_rects = []  # 選択済み矩形（PDFポイント単位）、複数蓄積

doc = fitz.open(pdf_path)
page = doc[0]
pix = page.get_pixmap(matrix=fitz.Matrix(SCALE, SCALE))
img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)

fig, ax = plt.subplots(figsize=(11, 15))
ax.imshow(img)
ax.set_title("ドラッグして白塗り範囲を選択（複数可）\n選んだら次のセルで適用")
ax.axis("off")

def on_select(eclick, erelease):
    px0 = min(eclick.xdata, erelease.xdata)
    py0 = min(eclick.ydata, erelease.ydata)
    px1 = max(eclick.xdata, erelease.xdata)
    py1 = max(eclick.ydata, erelease.ydata)
    selected_rects.append((px0/SCALE, py0/SCALE, px1/SCALE, py1/SCALE))
    rect = mpatches.Rectangle(
        (px0, py0), px1 - px0, py1 - py0,
        linewidth=1.5, edgecolor="red", facecolor="white", alpha=0.5
    )
    ax.add_patch(rect)
    fig.canvas.draw_idle()
    print(f"[{len(selected_rects)}] 追加 → 計 {len(selected_rects)} 箇所選択済み")

rs = RectangleSelector(
    ax, on_select,
    useblit=True, button=[1],
    minspanx=3, minspany=3,
    spancoords="pixels", interactive=False,
)
plt.tight_layout()
plt.show()
print("✅ ドラッグで選択 → 次のセルで適用 / やり直す場合: selected_rects.clear() してこのセルを再実行")

In [ ]:
# ============================================================
# セル1-a3: 手動白塗り（座標を指定して実行）
# ============================================================

# (x0, y0, x1, y1) のポイント座標で白塗りしたい矩形を列挙する
# 上の座標グリッド画像を見ながら指定してください
REDACT_RECTS = [
    # 例: (30, 280, 80, 300),   # 小節番号「3」
    # 例: (30, 680, 80, 700),   # 小節番号「15」
]

if not REDACT_RECTS:
    print("⚠️  REDACT_RECTS が空です。座標を入力してください")
else:
    doc = fitz.open(pdf_path)
    for page in doc:
        for rect in REDACT_RECTS:
            page.add_redact_annot(fitz.Rect(*rect), fill=(1, 1, 1))
        page.apply_redactions()

    clean_pdf_path = pdf_path.replace(".pdf", "_clean.pdf")
    doc.save(clean_pdf_path)

    # プレビュー
    pix = doc[0].get_pixmap(matrix=fitz.Matrix(2, 2))
    pix.save("/tmp/preview_clean.png")
    display(Image("/tmp/preview_clean.png"))
    print(f"\n✅ 保存完了 → {clean_pdf_path}")
    pdf_path = clean_pdf_path

In [ ]:
# ============================================================
# セル1-b: PDF → MusicXML（Audiveris）
# ============================================================

import os, subprocess, glob
from music21 import converter, stream

if not AUDIVERIS_BIN:
    raise RuntimeError("Audiveris が見つかりません。セル0-b を先に実行してください。")

OMR_DIR = "_omr_output/audiveris"
os.makedirs(OMR_DIR, exist_ok=True)
print("🎵 Audiveris で楽譜認識中...")

proc = subprocess.run(
    ["xvfb-run", AUDIVERIS_BIN,
     "-batch", "-export", "-output", OMR_DIR, "--", pdf_path],
    capture_output=True, text=True
)
if proc.returncode != 0:
    print("❌ Audiveris エラー:", proc.stderr[-500:])
else:
    print(proc.stdout[-300:] or "（出力なし）")

found_mxl = glob.glob(f"{OMR_DIR}/**/*.mxl", recursive=True)
found_xml  = glob.glob(f"{OMR_DIR}/**/*.xml",  recursive=True)
xml_paths  = found_mxl + found_xml

if not xml_paths:
    raise RuntimeError(f"MusicXML が生成されませんでした。出力フォルダ: {os.listdir(OMR_DIR)}")

if len(xml_paths) == 1:
    musicxml_path = xml_paths[0]
else:
    print(f"\n📎 {len(xml_paths)} ページをマージ中...")
    merged = stream.Score()
    for xp in xml_paths:
        for part in converter.parse(xp).parts:
            merged.append(part)
    musicxml_path = "_omr_output/merged.xml"
    merged.write("musicxml", fp=musicxml_path)

print(f"\n✅ MusicXML生成完了 → {musicxml_path}")

In [ ]:
# ============================================================
# セル1-b2: 楽譜を目視確認（verovio で SVG 表示）
# ============================================================

import subprocess, sys
from IPython.display import SVG, display

subprocess.run([sys.executable, "-m", "pip", "install", "verovio", "-q"], check=True)
import verovio

tk = verovio.toolkit()
tk.setOptions({"pageWidth": 1400, "scale": 35, "adjustPageHeight": 1})
tk.loadFile(musicxml_path)

page_count = tk.getPageCount()
print(f"ページ数: {page_count}")
for i in range(page_count):
    display(SVG(tk.renderToSVG(i + 1)))

In [ ]:
# ============================================================
# セル1-c: MusicXML を再生して認識精度を確認
# （初回はサウンドフォントのインストールで少し時間がかかります）
# ============================================================

import subprocess
from IPython.display import Audio, display
from music21 import converter

SOUNDFONT = "/usr/share/sounds/sf2/FluidR3_GM.sf2"
MIDI_PATH = "/tmp/preview.mid"
WAV_PATH  = "/tmp/preview.wav"

if not os.path.exists(SOUNDFONT):
    print("🔧 音声合成エンジンをインストール中（初回のみ）...")
    subprocess.run(["apt-get", "install", "-y", "fluidsynth", "fluid-soundfont-gm", "-q"], capture_output=True)

print("🎵 MusicXML を解析中...")
score = converter.parse(musicxml_path)
score.write("midi", fp=MIDI_PATH)

print("🔊 音声を生成中...")
res = subprocess.run(
    ["fluidsynth", "-ni", SOUNDFONT, MIDI_PATH, "-F", WAV_PATH, "-r", "44100"],
    capture_output=True, text=True
)
if res.returncode != 0:
    print("❌ 音声生成失敗:", res.stderr)
else:
    print("▶️  再生して内容を確認してください:")
    display(Audio(WAV_PATH))

---
## Step 2: 調弦を選択

In [ ]:
# ============================================================
# セル2: 調弦選択
# ============================================================

import ipywidgets as widgets
from IPython.display import display

tuning_widget = widgets.ToggleButtons(
    options=[
        ("本調子", "honchoshi"),
        ("二上り", "niagari"),
        ("三下り", "sansagari"),
    ],
    description="調弦:",
    button_style="info",
)
display(tuning_widget)

---
## Step 3: MusicXML → 三味線中間YAML

In [ ]:
# ============================================================
# セル3-a: 変換実行
# ============================================================

from shamisen_converter import (
    convert_musicxml, resolve_out_of_range,
    load_mapping, build_midi_to_position,
)

tuning = tuning_widget.value
print(f"調弦: {tuning_widget.label} ({tuning})")

result = convert_musicxml(
    musicxml_path=musicxml_path,
    mapping_path="shamisen_mapping.yaml",
    tuning=tuning,
)

total = len([n for n in result.notes if n.note_name != "rest"])
out_of_range = len([n for n in result.notes if n.out_of_range])
print(f"\n音符数: {total}")
print(f"音域外: {out_of_range} 件")
if result.warnings:
    print(f"警告数: {len(result.warnings)} 件")

In [ ]:
# ============================================================
# セル3-b: 音域外の処理（音域外がない場合はスキップ可）
# ============================================================

import ipywidgets as widgets
from IPython.display import display

mapping = load_mapping("shamisen_mapping.yaml")
midi_map = build_midi_to_position(mapping, tuning)
out_of_range_notes = [n for n in result.notes if n.out_of_range]

if not out_of_range_notes:
    print("✅ 音域外の音符はありません")
else:
    print(f"⚠️  {len(out_of_range_notes)} 件の音域外音符があります")

    policy_widget = widgets.RadioButtons(
        options=[
            ("すべてオクターブ上げて再変換", "up"),
            ("すべてオクターブ下げて再変換", "down"),
            ("すべて休符として扱う", "skip"),
            ("すべて未解決のまま残す", "keep"),
        ],
        description="処理方針:",
        layout=widgets.Layout(width="400px"),
    )
    apply_btn = widgets.Button(description="適用", button_style="success")

    def apply_policy(btn):
        policy = policy_widget.value
        for sn in result.notes:
            if not sn.out_of_range:
                continue
            if policy == "up":
                new_midi = sn.midi + 12
                if new_midi in midi_map:
                    s, p = midi_map[new_midi][0]
                    sn.string, sn.position, sn.midi = s, p, new_midi
                    sn.note_name += "(+8va)"
                    sn.out_of_range = False
                    sn.warning = f"オクターブ上げて解決: {sn.note_name}"
            elif policy == "down":
                new_midi = sn.midi - 12
                if new_midi in midi_map:
                    s, p = midi_map[new_midi][0]
                    sn.string, sn.position, sn.midi = s, p, new_midi
                    sn.note_name += "(-8va)"
                    sn.out_of_range = False
                    sn.warning = f"オクターブ下げて解決: {sn.note_name}"
            elif policy == "skip":
                sn.note_name = "rest"
                sn.out_of_range = False
                sn.warning = "音域外のためスキップ"
            # keep: そのまま
        print("✅ 音域外処理完了")

    apply_btn.on_click(apply_policy)
    display(policy_widget, apply_btn)

In [ ]:
# ============================================================
# セル3-c: 中間YAML出力
# ============================================================

from shamisen_converter import to_intermediate_yaml

OUTPUT_PATH = "shamisen_output.yaml"

yaml_str = to_intermediate_yaml(result)
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write(yaml_str)

print(f"✅ 変換完了 → {OUTPUT_PATH}")

---
## Step 4: ダウンロード

In [ ]:
# ============================================================
# セル4: ダウンロード
# ============================================================

from google.colab import files
files.download(OUTPUT_PATH)
print(f"⬇️  {OUTPUT_PATH} をダウンロードしました")

---
## （オプション）MusicXMLを直接アップロードして変換

PDFを使わずMusicXMLファイルから始める場合はこちら。

In [ ]:
# ============================================================
# オプション: MusicXMLを直接アップロード
# ============================================================

from google.colab import files

print("MusicXMLファイル（.xml / .mxl）を選択してください")
uploaded_xml = files.upload()
musicxml_path = list(uploaded_xml.keys())[0]
print(f"\n📄 アップロード: {musicxml_path}")
print("→ セル2（調弦選択）から続けてください")